# 🎤 Clone Your Own Voice (Free) — ClarityVoice

This notebook uses the free, open-source **XTTS-v2** model to make speech in **your** voice, from a short recording of you talking.

### How to use
1. **Turn on the free GPU:** menu **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**.
2. Run each cell **top to bottom** (click the ▶ on the left of each cell).
3. When asked, upload a **clear 15–30 second recording of your own voice** (WAV or MP3, no background noise — just you talking naturally).
4. Type what you want it to say, run the last cell, and it plays + downloads your cloned voice.

> First run downloads the model (~2 min). That's normal.

### 1. Check the GPU is on

In [ ]:
!nvidia-smi -L
# If this says 'command not found' or shows no GPU, go to Runtime > Change runtime type > T4 GPU.

### 2. Install the voice library (~1 min)

In [ ]:
!pip install -q coqui-tts
print("Installed. If the NEXT cell shows import errors, do Runtime > Restart session, then re-run from cell 3 (skip this install).")

### 3. Load the model

In [ ]:
import os, torch
os.environ["COQUI_TOS_AGREED"] = "1"

# Newer PyTorch defaults to weights_only=True, which breaks XTTS loading. Patch it:
_orig_load = torch.load
def _patched_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return _orig_load(*args, **kwargs)
torch.load = _patched_load

from TTS.api import TTS
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
print("Model loaded \u2705")

### 4. Upload YOUR voice recording (15–30s, clean)

In [ ]:
from google.colab import files
print("Upload a clear 15-30 second recording of you talking (WAV or MP3, no background noise).")
uploaded = files.upload()
sample_path = list(uploaded.keys())[0]
print("Using your voice sample:", sample_path)

### 5. Make it speak in your voice — multiple languages ✨
Each line below is one language. Edit the text, then run. This model supports **English (`en`)** and **Hindi (`hi`)**. Kannada is **not** supported here (ask for the Indic model for Kannada).

In [ ]:
from IPython.display import Audio, display

# One entry per language — edit the text freely. Supported here: "en" and "hi".
clips = {
    "en": "Hello, this is my own voice, cloned by AI. I can sound clear and professional.",
    "hi": "नमस्ते, यह मेरी अपनी आवाज़ है, जिसे एआई ने क्लोन किया है।",
}

for lang, text in clips.items():
    out = f"my_voice_{lang}.wav"
    tts.tts_to_file(text=text, speaker_wav=sample_path, language=lang, file_path=out)
    print(f"\n=== {lang} ===")
    display(Audio(out))
    files.download(out)

print("\nDone ✅  (Kannada 'kn' is not supported by this model — ask for the Indic model to add Kannada.)")